# Chapter 19 Companion Notebook: Modern Generative Models in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch19_Modern_Generative_Models.ipynb)

This notebook accompanies Chapter 19 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic marketing creative data and small PyTorch models so students can run the workflow in Colab without a paid API, a GPU, or an external dataset.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat markdown sections as short lecture notes and code sections as live demos. If a section runs slowly, keep `FAST_MODE = True`, reduce `N_CONCEPTS`, reduce `EPOCHS_DIFFUSION`, or reduce `N_SAMPLE_EVAL` in the setup cell.

## Why this matters (business framing)

Modern generative AI is useful in business analytics when it becomes controllable, evaluable, and repeatable. A marketing team does not only need attractive outputs. It needs outputs that match the brief, respect brand rules, avoid unsupported claims, preserve approved elements when editing, and leave a record that another person can audit.

This notebook follows the practical logic of Chapter 19. We will build a small diffusion model in a compact latent space, visualize the forward noising process, train a denoiser to predict injected noise, sample new candidates with different step counts, use conditioning and guidance to steer outputs, demonstrate editing from an intermediate noise level, simulate inpainting-style constraints by locking approved feature dimensions, connect the generator to a retrieval and evaluation workflow, and finish with provenance and governance artifacts. The goal is not to produce photorealistic images. The goal is to understand the durable workflow behind diffusion and foundation-model systems.

## Agenda

1. Setup and reproducibility
2. Synthetic marketing creative concept library
3. Time-respecting split, latent representation, and retrieval setup
4. Forward diffusion as controlled information loss
5. Reverse denoising as a supervised learning problem
6. Sampling speed and the quality frontier
7. Conditioning and classifier-free guidance
8. Latent diffusion as compression, generation, and reconstruction
9. Editing workflows: image-to-image logic and masked constraints
10. Foundation-model ecosystem: retrieval, conditioning, generation, evaluation, and provenance
11. Business readiness checks and exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to describe a diffusion timestep as a noise index, implement a forward noising process, train a small denoiser to predict injected noise, sample from a conditional diffusion model, explain the tradeoff between step count and quality, compare guidance strengths using adherence and diversity metrics, explain why latent diffusion reduces computational cost, simulate editing by starting denoising from an intermediate noise level, build a simple retrieval-augmented conditioning package, evaluate generated candidates with quality gates, and write a lightweight provenance record for a generative workflow.

## Connection map

Chapter 18 introduced VAEs and GANs as classic generative models. Chapter 19 changes the generation story from one-shot sampling to iterative refinement. A diffusion model learns to reverse a fixed corruption process one step at a time. In production, this generator is usually only one module inside a larger foundation-model ecosystem that also includes conditioning, retrieval, evaluation, human review, and provenance logging.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================
import os
import sys
import math
import json
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 19
FAST_MODE = True
N_CONCEPTS = 3600 if FAST_MODE else 7000
BATCH_SIZE = 256
LATENT_DIMS = 2
T_STEPS = 40
EPOCHS_DIFFUSION = 380 if FAST_MODE else 750
COND_DROPOUT = 0.12
N_SAMPLE_EVAL = 500 if FAST_MODE else 1000

OUTPUT_DIR = Path("ch19_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
try:
    torch.set_num_threads(1)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"FAST_MODE: {FAST_MODE}")
print(f"Synthetic creative concepts: {N_CONCEPTS:,}")

## Utility functions

These helpers keep the main sections focused on modeling and decision logic. A diffusion workflow has many moving pieces, so the notebook uses small functions for plotting, encoding, decoding, scoring, and evaluation.

In [ ]:
# ============================================================
# Utility functions for display, plotting, metrics, and files
# ============================================================

def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=device)


def plot_history(history_df, title="Training history"):
    plt.figure(figsize=(8, 4.5))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Noise prediction MSE")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


def plot_latent(real_points, generated_points=None, title="Latent space", real_label="Real", generated_label="Generated", max_points=900):
    rng = np.random.default_rng(SEED)
    plt.figure(figsize=(6.5, 5.3))
    r_idx = rng.choice(len(real_points), size=min(max_points, len(real_points)), replace=False)
    plt.scatter(real_points[r_idx, 0], real_points[r_idx, 1], s=13, alpha=0.35, label=real_label)
    if generated_points is not None and len(generated_points) > 0:
        g_idx = rng.choice(len(generated_points), size=min(max_points, len(generated_points)), replace=False)
        plt.scatter(generated_points[g_idx, 0], generated_points[g_idx, 1], s=13, alpha=0.38, label=generated_label)
    plt.title(title)
    plt.xlabel("Latent dimension 1")
    plt.ylabel("Latent dimension 2")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


def nearest_neighbor_mean_distance(query_points, reference_points):
    nn = NearestNeighbors(n_neighbors=1).fit(reference_points)
    distances, _ = nn.kneighbors(query_points)
    return float(distances.mean())


def coverage_rate(reference_points, generated_points, threshold):
    nn = NearestNeighbors(n_neighbors=1).fit(generated_points)
    distances, _ = nn.kneighbors(reference_points)
    return float(np.mean(distances[:, 0] <= threshold))


def generated_diversity(generated_points):
    if len(generated_points) < 2:
        return np.nan
    nn = NearestNeighbors(n_neighbors=2).fit(generated_points)
    distances, _ = nn.kneighbors(generated_points)
    return float(distances[:, 1].mean())


def real_real_radius(reference_points, percentile=90):
    if len(reference_points) < 3:
        return 0.25
    nn = NearestNeighbors(n_neighbors=2).fit(reference_points)
    distances, _ = nn.kneighbors(reference_points)
    return float(np.percentile(distances[:, 1], percentile))


def write_json(path, payload):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

## 2. Synthetic marketing creative concept library

We use a synthetic creative library because this notebook is meant to be safe, shareable, and runnable in class. Each row represents an approved or not-yet-approved marketing concept. The concept has structured conditions, short text, latent design features, and simple business quality scores. In a real workflow, the rows might come from past campaign assets, concept tests, review logs, brand guidelines, or a creative asset management system.

In [ ]:
# ============================================================
# 2.1 Synthetic creative concept library
# ============================================================
PRODUCTS = ["snack", "skincare", "fitness_app", "banking"]
AUDIENCES = ["value_seekers", "premium_loyalists", "new_customers", "sustainability_minded"]
STYLES = ["minimal", "playful", "premium", "instructional", "social_proof"]
OBJECTIVES = ["awareness", "conversion", "retention"]

FEATURE_COLS = [
    "brightness", "warmth", "visual_density", "whitespace", "product_focus", "human_presence",
    "price_cue", "sustainability_cue", "luxury_cue", "urgency_cue", "copy_complexity", "claim_strength"
]

base = np.array([0.55, 0.52, 0.48, 0.50, 0.55, 0.45, 0.35, 0.30, 0.35, 0.35, 0.45, 0.35])

product_effects = {
    "snack": np.array([0.10, 0.15, 0.08, -0.05, 0.10, 0.05, 0.10, -0.05, -0.08, 0.10, -0.05, 0.05]),
    "skincare": np.array([0.05, 0.02, -0.10, 0.12, 0.10, 0.08, -0.02, 0.08, 0.08, -0.08, 0.02, 0.03]),
    "fitness_app": np.array([0.02, 0.05, 0.02, 0.02, -0.05, 0.15, -0.05, 0.00, -0.05, 0.12, 0.08, 0.10]),
    "banking": np.array([-0.04, -0.05, -0.08, 0.10, 0.02, -0.02, -0.03, -0.02, 0.10, -0.10, 0.12, 0.08]),
}

audience_effects = {
    "value_seekers": np.array([0.03, 0.02, 0.08, -0.06, 0.03, -0.02, 0.22, -0.05, -0.12, 0.14, -0.03, 0.00]),
    "premium_loyalists": np.array([-0.02, -0.02, -0.08, 0.12, 0.05, 0.02, -0.12, 0.00, 0.24, -0.08, 0.00, 0.03]),
    "new_customers": np.array([0.08, 0.04, 0.05, -0.04, 0.00, 0.10, 0.04, 0.00, -0.05, 0.08, 0.12, 0.05]),
    "sustainability_minded": np.array([0.03, -0.02, -0.05, 0.08, 0.03, 0.05, -0.05, 0.25, 0.04, -0.05, 0.04, 0.00]),
}

style_effects = {
    "minimal": np.array([0.02, -0.02, -0.22, 0.28, 0.10, -0.04, -0.08, 0.03, 0.05, -0.12, -0.10, -0.02]),
    "playful": np.array([0.17, 0.14, 0.16, -0.18, -0.02, 0.08, 0.06, 0.00, -0.12, 0.12, -0.02, 0.04]),
    "premium": np.array([-0.03, -0.03, -0.10, 0.16, 0.05, 0.00, -0.15, 0.00, 0.30, -0.10, -0.02, 0.00]),
    "instructional": np.array([-0.06, -0.03, 0.05, -0.02, 0.00, -0.04, -0.03, 0.02, -0.02, -0.04, 0.24, 0.05]),
    "social_proof": np.array([0.02, 0.02, 0.10, -0.08, -0.08, 0.20, 0.00, 0.00, -0.02, 0.08, 0.06, 0.06]),
}

objective_effects = {
    "awareness": np.array([0.08, 0.04, 0.06, -0.06, -0.06, 0.05, 0.00, 0.03, 0.00, 0.04, -0.02, 0.02]),
    "conversion": np.array([0.03, 0.02, 0.08, -0.08, 0.10, 0.00, 0.13, -0.02, -0.02, 0.18, 0.02, 0.05]),
    "retention": np.array([-0.02, -0.03, -0.04, 0.08, 0.02, 0.05, -0.04, 0.05, 0.08, -0.12, 0.08, 0.03]),
}

style_words = {
    "minimal": "clean whitespace simple calm layout",
    "playful": "bright playful energetic friendly shapes",
    "premium": "elegant refined premium luxury quiet confidence",
    "instructional": "step by step explanation helpful tutorial",
    "social_proof": "review testimonials people community trust",
}
objective_words = {
    "awareness": "introduce brand story recognition reach",
    "conversion": "offer trial purchase urgency sign up",
    "retention": "loyalty repeat use relationship service",
}


def business_scores_from_features(feat_df, product, audience, style, objective):
    """Reusable scoring rule for synthetic brand fit and compliance risk."""
    x = feat_df[FEATURE_COLS].to_numpy()
    brand = 0.30 + 0.25 * x[:, 3] + 0.22 * x[:, 4] + 0.15 * x[:, 8] + 0.12 * x[:, 7] - 0.18 * x[:, 10] - 0.12 * x[:, 9]
    if style == "playful":
        brand += 0.07 if product == "snack" else -0.05
    if style == "instructional":
        brand += 0.06 if product in ["banking", "fitness_app"] else -0.03
    if style == "minimal":
        brand += 0.06 if product in ["skincare", "banking"] else -0.01
    if audience == "sustainability_minded":
        brand += 0.08 * x[:, 7]

    risk = 0.10 + 0.30 * x[:, 11] + 0.20 * x[:, 9] + 0.10 * x[:, 10]
    if product in ["banking", "fitness_app", "skincare"]:
        risk += 0.08 * x[:, 11]
    if objective == "conversion":
        risk += 0.06 * x[:, 9]
    return np.clip(brand, 0, 1), np.clip(risk, 0, 1)


def generate_creative_library(n=N_CONCEPTS, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        month = int(rng.integers(1, 25))
        product = rng.choice(PRODUCTS, p=[0.28, 0.25, 0.23, 0.24])
        audience = rng.choice(AUDIENCES)
        if product == "banking":
            style = rng.choice(STYLES, p=[0.25, 0.05, 0.25, 0.30, 0.15])
        elif product == "skincare":
            style = rng.choice(STYLES, p=[0.30, 0.08, 0.25, 0.12, 0.25])
        elif product == "snack":
            style = rng.choice(STYLES, p=[0.08, 0.42, 0.08, 0.12, 0.30])
        else:
            style = rng.choice(STYLES, p=[0.15, 0.20, 0.10, 0.25, 0.30])
        objective = rng.choice(OBJECTIVES, p=[0.35, 0.35, 0.30])

        mu = base + product_effects[product] + audience_effects[audience] + style_effects[style] + objective_effects[objective]
        x = np.clip(mu + rng.normal(0, 0.075, len(FEATURE_COLS)), 0.02, 0.98)
        temp_feat = pd.DataFrame([x], columns=FEATURE_COLS)
        brand_fit, compliance_risk = business_scores_from_features(temp_feat, product, audience, style, objective)
        brand_fit = float(np.clip(brand_fit[0] + rng.normal(0, 0.035), 0, 1))
        compliance_risk = float(np.clip(compliance_risk[0] + rng.normal(0, 0.025), 0, 1))
        expected_ctr = float(np.clip(0.02 + 0.08 * x[0] + 0.07 * x[1] + 0.06 * x[5] + 0.08 * x[9] + 0.04 * x[7] + rng.normal(0, 0.01), 0.005, 0.35))
        approved = bool((brand_fit > 0.58) and (compliance_risk < 0.55))
        concept_text = (
            f"{style.replace('_', ' ')} {product.replace('_', ' ')} creative for "
            f"{audience.replace('_', ' ')} focused on {objective}. "
            f"Visual cues: {style_words[style]}. Campaign intent: {objective_words[objective]}."
        )
        row = {
            "concept_id": f"C{i:05d}",
            "as_of_month": month,
            "product_category": product,
            "audience": audience,
            "style": style,
            "objective": objective,
            "concept_text": concept_text,
            "brand_fit": brand_fit,
            "compliance_risk": compliance_risk,
            "expected_ctr": expected_ctr,
            "approved": approved,
        }
        row.update({FEATURE_COLS[j]: float(x[j]) for j in range(len(FEATURE_COLS))})
        rows.append(row)
    return pd.DataFrame(rows)


concepts_df = generate_creative_library()

print_section("Creative concept library")
print(concepts_df.shape)
display(concepts_df.head(5))

summary = (
    concepts_df.groupby(["product_category", "style"])
    .agg(n=("concept_id", "size"), approval_rate=("approved", "mean"), avg_brand_fit=("brand_fit", "mean"), avg_risk=("compliance_risk", "mean"))
    .reset_index()
    .sort_values(["product_category", "style"])
)
display(summary.head(12))

## 3. Time-respecting split, latent representation, and retrieval setup

A production system should not train on future review behavior and then claim to represent the past. We split the synthetic library by `as_of_month`: earlier months are used for fitting, and later months are used for validation. We then compress the structured creative features into a two-dimensional latent space using PCA. This is a lightweight classroom proxy for the latent representation used in latent diffusion systems.

In [ ]:
# ============================================================
# 3.1 Time-respecting split, preprocessing, and PCA latent space
# ============================================================
train_df = concepts_df[concepts_df["as_of_month"] <= 18].copy()
test_df = concepts_df[concepts_df["as_of_month"] > 18].copy()

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS])
X_test = scaler.transform(test_df[FEATURE_COLS])

pca = PCA(n_components=LATENT_DIMS, random_state=SEED)
Z_train = pca.fit_transform(X_train).astype(np.float32)
Z_test = pca.transform(X_test).astype(np.float32)

cond_cols = ["product_category", "audience", "style", "objective"]
try:
    cond_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    cond_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

C_train = cond_encoder.fit_transform(train_df[cond_cols]).astype(np.float32)
C_test = cond_encoder.transform(test_df[cond_cols]).astype(np.float32)
COND_DIM = C_train.shape[1]

split_summary = pd.DataFrame([
    {"split": "train", "months": "1 to 18", "n": len(train_df)},
    {"split": "test", "months": "19 to 24", "n": len(test_df)},
])
print_section("Time split and latent representation")
display(split_summary)
print(f"Condition vector dimension: {COND_DIM}")
print(f"PCA explained variance ratio: {np.round(pca.explained_variance_ratio_, 3)}")
print(f"Total variance captured by {LATENT_DIMS} latent dimensions: {pca.explained_variance_ratio_.sum():.3f}")

plot_latent(Z_train, Z_test, title="Train versus test concepts in PCA latent space", real_label="Train", generated_label="Test")

In [ ]:
# ============================================================
# 3.2 Text retrieval setup for a foundation-model workflow
# ============================================================
# Retrieval is used later to fetch approved examples and constraints.
# We use TF-IDF instead of an API-based embedding model to keep the notebook fully offline.
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
text_matrix = vectorizer.fit_transform(concepts_df["concept_text"])
retriever = NearestNeighbors(n_neighbors=6, metric="cosine")
retriever.fit(text_matrix)

print(f"TF-IDF matrix shape: {text_matrix.shape}")
print("The retrieval component is ready for use in Section 10.")

## 4. Forward diffusion as controlled information loss

In diffusion, the timestep is a noise index, not calendar time. The forward process starts with clean data at `t = 0` and gradually adds noise until `t = T`. Because the forward process is fixed, we know the exact noise that was added. That is what turns training into a supervised denoising problem.

In [ ]:
# ============================================================
# 4.1 Diffusion noise schedule
# ============================================================
betas = torch.linspace(1e-4, 0.035, T_STEPS, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

schedule_df = pd.DataFrame({
    "timestep": np.arange(T_STEPS),
    "beta": betas.detach().cpu().numpy(),
    "signal_weight_sqrt_alpha_bar": torch.sqrt(alpha_bars).detach().cpu().numpy(),
    "noise_weight_sqrt_one_minus_alpha_bar": torch.sqrt(1 - alpha_bars).detach().cpu().numpy(),
})

display(schedule_df.iloc[[0, 5, 10, 20, 30, 39]].round(4))

plt.figure(figsize=(8, 4.5))
plt.plot(schedule_df["timestep"], schedule_df["signal_weight_sqrt_alpha_bar"], label="Signal weight")
plt.plot(schedule_df["timestep"], schedule_df["noise_weight_sqrt_one_minus_alpha_bar"], label="Noise weight")
plt.title("Forward diffusion schedule: signal shrinks and noise grows")
plt.xlabel("Diffusion timestep t")
plt.ylabel("Weight")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# 4.2 Controlled corruption of real latent concepts
# ============================================================
def q_sample(x0, t, noise):
    """Add noise to x0 at timestep t using the fixed forward process."""
    sqrt_ab = torch.sqrt(alpha_bars[t]).view(-1, 1)
    sqrt_om = torch.sqrt(1 - alpha_bars[t]).view(-1, 1)
    return sqrt_ab * x0 + sqrt_om * noise

rng = np.random.default_rng(SEED)
subset_idx = rng.choice(len(Z_train), size=450, replace=False)
base_points = torch.tensor(Z_train[subset_idx], dtype=torch.float32, device=device)
shared_noise = torch.randn_like(base_points)

for t_value in [0, 10, 25, 39]:
    t_batch = torch.full((len(base_points),), t_value, dtype=torch.long, device=device)
    noisy_points = q_sample(base_points, t_batch, shared_noise).detach().cpu().numpy()
    plot_latent(
        Z_train[subset_idx],
        noisy_points,
        title=f"Forward corruption at timestep t = {t_value}",
        real_label="Clean latent concepts",
        generated_label="Noisy latent concepts",
        max_points=450,
    )

## 5. Reverse denoising as a supervised learning problem

The model receives three inputs: the noisy latent vector, the timestep, and an optional condition vector. It predicts the noise that was added. During training, we sometimes remove the condition vector. This small trick creates an unconditional path and a conditional path, which lets us use classifier-free guidance during sampling.

In [ ]:
# ============================================================
# 5.1 Conditional denoiser network
# ============================================================
class ConditionalDenoiser(nn.Module):
    def __init__(self, x_dim, cond_dim, n_steps, time_dim=16, hidden=96):
        super().__init__()
        self.time_emb = nn.Embedding(n_steps, time_dim)
        self.net = nn.Sequential(
            nn.Linear(x_dim + cond_dim + time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden // 2),
            nn.SiLU(),
            nn.Linear(hidden // 2, x_dim),
        )

    def forward(self, x_t, t, cond):
        t_emb = self.time_emb(t)
        return self.net(torch.cat([x_t, t_emb, cond], dim=1))


model = ConditionalDenoiser(LATENT_DIMS, COND_DIM, T_STEPS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

train_loader = DataLoader(
    TensorDataset(torch.tensor(Z_train, dtype=torch.float32), torch.tensor(C_train, dtype=torch.float32)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_x = torch.tensor(Z_test[: min(512, len(Z_test))], dtype=torch.float32, device=device)
val_c = torch.tensor(C_test[: min(512, len(C_test))], dtype=torch.float32, device=device)

print(model)

In [ ]:
# ============================================================
# 5.2 Train the denoiser to predict injected noise
# ============================================================
set_seed(SEED)
model.train()
history = []
start_time = time.time()

for epoch in range(1, EPOCHS_DIFFUSION + 1):
    total_loss = 0.0
    total_n = 0
    for xb, cb in train_loader:
        xb = xb.to(device)
        cb = cb.to(device)
        batch_n = xb.size(0)

        t = torch.randint(0, T_STEPS, (batch_n,), device=device).long()
        noise = torch.randn_like(xb)
        x_t = q_sample(xb, t, noise)

        # Classifier-free guidance training: randomly drop the condition.
        drop_mask = (torch.rand(batch_n, device=device) < COND_DROPOUT).float().view(-1, 1)
        cb_train = cb * (1 - drop_mask)

        pred_noise = model(x_t, t, cb_train)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_n
        total_n += batch_n

    if epoch == 1 or epoch % 50 == 0 or epoch == EPOCHS_DIFFUSION:
        model.eval()
        with torch.no_grad():
            b = val_x.size(0)
            t_val = torch.randint(0, T_STEPS, (b,), device=device).long()
            noise_val = torch.randn_like(val_x)
            x_val_t = q_sample(val_x, t_val, noise_val)
            pred_val = model(x_val_t, t_val, val_c)
            val_loss = F.mse_loss(pred_val, noise_val).item()
        history.append({"epoch": epoch, "train_loss": total_loss / total_n, "val_loss": val_loss})
        model.train()

history_df = pd.DataFrame(history)
print(f"Training time: {time.time() - start_time:.1f} seconds")
display(history_df.round(4))
plot_history(history_df, title="Diffusion denoiser training history")

## 6. Sampling speed and the quality frontier

Generation runs the learned denoising process backward. More denoising steps usually give the model more opportunities to refine the sample, but the improvements flatten after a point. This creates a speed-quality frontier. The code below uses a deterministic DDIM-style sampler for a compact classroom demonstration.

In [ ]:
# ============================================================
# 6.1 Sampling, decoding, and evaluation helpers
# ============================================================
def make_condition(product, audience, style, objective):
    request_df = pd.DataFrame([{
        "product_category": product,
        "audience": audience,
        "style": style,
        "objective": objective,
    }])
    return cond_encoder.transform(request_df[cond_cols]).astype(np.float32)[0]


@torch.no_grad()
def sample_ddim(cond_np, n_samples=N_SAMPLE_EVAL, guidance_scale=1.5, num_steps=T_STEPS, seed=None):
    """Deterministic DDIM-style sampler with classifier-free guidance."""
    model.eval()
    if seed is not None:
        gen = torch.Generator(device=device)
        gen.manual_seed(seed)
        x = torch.randn((n_samples, LATENT_DIMS), generator=gen, device=device)
    else:
        x = torch.randn((n_samples, LATENT_DIMS), device=device)

    if cond_np.ndim == 1:
        cond_np = np.repeat(cond_np[None, :], n_samples, axis=0)
    cond = torch.tensor(cond_np, dtype=torch.float32, device=device)
    if cond.size(0) != n_samples:
        cond = cond.repeat(n_samples, 1)
    cond_zero = torch.zeros_like(cond)

    raw_schedule = np.linspace(T_STEPS - 1, 0, num_steps, dtype=int)
    schedule = []
    for step in raw_schedule:
        step = int(step)
        if len(schedule) == 0 or step != schedule[-1]:
            schedule.append(step)

    for i, t_int in enumerate(schedule):
        t_batch = torch.full((n_samples,), t_int, dtype=torch.long, device=device)
        eps_uncond = model(x, t_batch, cond_zero)
        eps_cond = model(x, t_batch, cond)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        alpha_bar_t = alpha_bars[t_int]
        x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)

        if i == len(schedule) - 1:
            x = x0_pred
        else:
            prev_t = schedule[i + 1]
            alpha_bar_prev = alpha_bars[prev_t]
            x = torch.sqrt(alpha_bar_prev) * x0_pred + torch.sqrt(1 - alpha_bar_prev) * eps

    return x.detach().cpu().numpy()


def decode_latents(z_points):
    """Decode PCA latent samples back to the original synthetic feature scale."""
    x_scaled = pca.inverse_transform(z_points)
    x_original = scaler.inverse_transform(x_scaled)
    return pd.DataFrame(np.clip(x_original, 0, 1), columns=FEATURE_COLS)


def reference_subset(product, style, audience=None, objective=None):
    mask = (test_df["product_category"] == product) & (test_df["style"] == style)
    if audience is not None:
        mask = mask & (test_df["audience"] == audience)
    if objective is not None:
        mask = mask & (test_df["objective"] == objective)
    if mask.sum() < 30:
        mask = (test_df["product_category"] == product) & (test_df["style"] == style)
    if mask.sum() < 30:
        mask = test_df["product_category"] == product
    return Z_test[mask.to_numpy()]


product_clf = LogisticRegression(max_iter=500).fit(Z_train, train_df["product_category"])
style_clf = LogisticRegression(max_iter=500).fit(Z_train, train_df["style"])

print(f"Product classifier validation accuracy on latent space: {product_clf.score(Z_test, test_df['product_category']):.3f}")
print(f"Style classifier validation accuracy on latent space: {style_clf.score(Z_test, test_df['style']):.3f}")

In [ ]:
# ============================================================
# 6.2 Speed-quality frontier under a fixed request
# ============================================================
REQUEST = {
    "product": "skincare",
    "audience": "sustainability_minded",
    "style": "minimal",
    "objective": "retention",
}
request_cond = make_condition(**REQUEST)
Z_ref = reference_subset(REQUEST["product"], REQUEST["style"], REQUEST["audience"], REQUEST["objective"])
ref_radius = real_real_radius(Z_ref)

frontier_rows = []
frontier_samples = {}
for num_steps in [8, 16, 30, 40]:
    t0 = time.time()
    z_gen = sample_ddim(request_cond, n_samples=N_SAMPLE_EVAL, guidance_scale=1.6, num_steps=num_steps, seed=SEED + num_steps)
    elapsed = time.time() - t0
    frontier_samples[num_steps] = z_gen
    prod_pred = product_clf.predict(z_gen)
    style_pred = style_clf.predict(z_gen)
    frontier_rows.append({
        "num_denoising_steps": num_steps,
        "elapsed_seconds": elapsed,
        "mean_nn_distance_to_reference": nearest_neighbor_mean_distance(z_gen, Z_ref),
        "reference_coverage_rate": coverage_rate(Z_ref, z_gen, ref_radius),
        "generated_diversity": generated_diversity(z_gen),
        "product_adherence_rate": float(np.mean(prod_pred == REQUEST["product"])),
        "style_adherence_rate": float(np.mean(style_pred == REQUEST["style"])),
    })

frontier_df = pd.DataFrame(frontier_rows)
display(frontier_df.round(4))

plt.figure(figsize=(7, 4.5))
plt.plot(frontier_df["num_denoising_steps"], frontier_df["mean_nn_distance_to_reference"], marker="o")
plt.title("Speed-quality frontier: more steps versus reference distance")
plt.xlabel("Number of denoising steps")
plt.ylabel("Mean nearest-neighbor distance to reference")
plt.grid(alpha=0.25)
plt.show()

plot_latent(Z_ref, frontier_samples[30], title="Generated samples for the request versus matching test references", real_label="Reference test concepts", generated_label="Generated concepts")

## 7. Conditioning and classifier-free guidance

Conditioning tells the model what kind of output is requested. Guidance controls how strongly the model follows that condition. Higher guidance can improve adherence, but it can also reduce diversity or push samples into less natural regions. This is why guidance strength should be treated as a policy setting, not merely a visual preference.

In [ ]:
# ============================================================
# 7.1 Guidance strength experiment
# ============================================================
guidance_rows = []
guidance_samples = {}
for guidance_scale in [0.0, 0.8, 1.6, 3.0]:
    z_gen = sample_ddim(request_cond, n_samples=N_SAMPLE_EVAL, guidance_scale=guidance_scale, num_steps=30, seed=SEED + int(guidance_scale * 10) + 101)
    guidance_samples[guidance_scale] = z_gen
    feat_gen = decode_latents(z_gen)
    brand_fit, risk = business_scores_from_features(feat_gen, **REQUEST)
    prod_pred = product_clf.predict(z_gen)
    style_pred = style_clf.predict(z_gen)
    guidance_rows.append({
        "guidance_scale": guidance_scale,
        "mean_nn_distance_to_reference": nearest_neighbor_mean_distance(z_gen, Z_ref),
        "reference_coverage_rate": coverage_rate(Z_ref, z_gen, ref_radius),
        "generated_diversity": generated_diversity(z_gen),
        "product_adherence_rate": float(np.mean(prod_pred == REQUEST["product"])),
        "style_adherence_rate": float(np.mean(style_pred == REQUEST["style"])),
        "avg_brand_fit": float(np.mean(brand_fit)),
        "avg_compliance_risk": float(np.mean(risk)),
    })

guidance_df = pd.DataFrame(guidance_rows)
display(guidance_df.round(4))

plt.figure(figsize=(7.5, 4.5))
plt.plot(guidance_df["guidance_scale"], guidance_df["product_adherence_rate"], marker="o", label="Product adherence")
plt.plot(guidance_df["guidance_scale"], guidance_df["style_adherence_rate"], marker="o", label="Style adherence")
plt.plot(guidance_df["guidance_scale"], guidance_df["generated_diversity"], marker="o", label="Generated diversity")
plt.title("Guidance strength changes adherence and diversity")
plt.xlabel("Guidance scale")
plt.ylabel("Metric value")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# 7.2 Compare low-guidance and high-guidance samples visually
# ============================================================
plot_latent(
    Z_ref,
    guidance_samples[0.8],
    title="Lower guidance: more freedom, weaker steering",
    real_label="Reference test concepts",
    generated_label="Generated concepts",
)
plot_latent(
    Z_ref,
    guidance_samples[3.0],
    title="Higher guidance: stronger steering, possible concentration",
    real_label="Reference test concepts",
    generated_label="Generated concepts",
)

## 8. Latent diffusion as compression, generation, and reconstruction

Latent diffusion generates in a compressed representation rather than directly in the original data space. In this notebook, PCA plays the role of a simple encoder and decoder. Real systems often use learned encoders and decoders, but the managerial idea is the same: compressed generation can be cheaper, faster, and easier to control, while reconstruction quality still needs to be checked.

In [ ]:
# ============================================================
# 8.1 Decode generated latent samples back to creative features
# ============================================================
z_request = guidance_samples[1.6]
features_generated = decode_latents(z_request)
brand_fit_gen, risk_gen = business_scores_from_features(features_generated, **REQUEST)
features_generated["brand_fit"] = brand_fit_gen
features_generated["compliance_risk"] = risk_gen

feature_summary = features_generated[FEATURE_COLS + ["brand_fit", "compliance_risk"]].agg(["mean", "std"]).T.reset_index()
feature_summary.columns = ["feature", "generated_mean", "generated_std"]
display(feature_summary.round(3))

# Reconstruction error tells us how much detail was lost by the PCA proxy.
X_train_reconstructed = scaler.inverse_transform(pca.inverse_transform(Z_train))
X_train_original = train_df[FEATURE_COLS].to_numpy()
pca_rmse = float(np.sqrt(np.mean((X_train_reconstructed - X_train_original) ** 2)))
print(f"PCA proxy reconstruction RMSE on original feature scale: {pca_rmse:.4f}")

In [ ]:
# ============================================================
# 8.2 Candidate-level generated profiles
# ============================================================
candidates_preview = features_generated.sample(8, random_state=SEED).copy()
candidates_preview.insert(0, "candidate_id", [f"G{i:03d}" for i in range(len(candidates_preview))])
preview_cols = [
    "candidate_id", "brightness", "whitespace", "product_focus", "sustainability_cue",
    "luxury_cue", "urgency_cue", "copy_complexity", "claim_strength", "brand_fit", "compliance_risk"
]
display(candidates_preview[preview_cols].round(3))

## 9. Editing workflows: image-to-image logic and masked constraints

Image-to-image diffusion begins from an existing asset rather than pure noise. A small amount of noise preserves much of the original structure; a larger amount gives the model more freedom to change the asset. Inpainting and outpainting can be understood as masked generation: some regions or attributes are locked, while others are regenerated under the condition.

In [ ]:
# ============================================================
# 9.1 Image-to-image style edit in latent space
# ============================================================
@torch.no_grad()
def edit_from_latent(x0_np, cond_np, strength=0.45, guidance_scale=1.6, num_steps=None, seed=None):
    """Start from an existing latent point, add noise at t_start, then denoise under a new condition."""
    model.eval()
    t_start = int(np.clip(round(strength * (T_STEPS - 1)), 1, T_STEPS - 1))
    x0 = torch.tensor(x0_np.reshape(1, -1), dtype=torch.float32, device=device)
    if seed is not None:
        torch.manual_seed(seed)
    noise = torch.randn_like(x0)
    t_batch = torch.tensor([t_start], dtype=torch.long, device=device)
    x = q_sample(x0, t_batch, noise)

    cond = torch.tensor(cond_np.reshape(1, -1), dtype=torch.float32, device=device)
    cond_zero = torch.zeros_like(cond)

    total_steps = num_steps or (t_start + 1)
    raw_schedule = np.linspace(t_start, 0, total_steps, dtype=int)
    schedule = []
    for step in raw_schedule:
        step = int(step)
        if len(schedule) == 0 or step != schedule[-1]:
            schedule.append(step)

    for i, t_int in enumerate(schedule):
        t = torch.full((1,), t_int, dtype=torch.long, device=device)
        eps_uncond = model(x, t, cond_zero)
        eps_cond = model(x, t, cond)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        alpha_bar_t = alpha_bars[t_int]
        x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)
        if i == len(schedule) - 1:
            x = x0_pred
        else:
            prev_t = schedule[i + 1]
            alpha_bar_prev = alpha_bars[prev_t]
            x = torch.sqrt(alpha_bar_prev) * x0_pred + torch.sqrt(1 - alpha_bar_prev) * eps
    return x.detach().cpu().numpy()[0]

# Choose one existing test concept and edit it toward a different request.
original_row = test_df.sample(1, random_state=SEED).iloc[0]
original_index = test_df.index.get_loc(original_row.name)
original_latent = Z_test[original_index]
edit_request = {
    "product": "snack",
    "audience": "value_seekers",
    "style": "playful",
    "objective": "conversion",
}
edit_cond = make_condition(**edit_request)

edit_rows = []
edit_points = []
for strength in [0.15, 0.45, 0.75]:
    z_edit = edit_from_latent(original_latent, edit_cond, strength=strength, guidance_scale=1.7, seed=SEED + int(strength * 100))
    edit_points.append(z_edit)
    decoded = decode_latents(z_edit.reshape(1, -1))
    brand, risk = business_scores_from_features(decoded, **edit_request)
    edit_rows.append({
        "edit_strength": strength,
        "distance_from_original_latent": float(np.linalg.norm(z_edit - original_latent)),
        "predicted_product": product_clf.predict(z_edit.reshape(1, -1))[0],
        "predicted_style": style_clf.predict(z_edit.reshape(1, -1))[0],
        "brand_fit": float(brand[0]),
        "compliance_risk": float(risk[0]),
    })

print("Original concept:")
display(original_row[["concept_id", "product_category", "audience", "style", "objective", "brand_fit", "compliance_risk"]].to_frame().T)
display(pd.DataFrame(edit_rows).round(3))

edit_points = np.vstack(edit_points)
plot_latent(
    original_latent.reshape(1, -1),
    edit_points,
    title="Editing from an existing latent concept",
    real_label="Original concept",
    generated_label="Edited versions",
    max_points=10,
)

In [ ]:
# ============================================================
# 9.2 Masked feature locking as an inpainting-style constraint
# ============================================================
# We simulate a local edit by locking approved attributes and replacing only editable attributes.
original_features = pd.DataFrame([original_row[FEATURE_COLS].astype(float).to_numpy()], columns=FEATURE_COLS)
strong_edit_features = decode_latents(edit_points[-1].reshape(1, -1))

locked_features = ["product_focus", "whitespace", "luxury_cue"]
masked_edit = strong_edit_features.copy()
for col in locked_features:
    masked_edit[col] = original_features[col].values

mask_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "original": original_features.iloc[0].values,
    "strong_edit_before_mask": strong_edit_features.iloc[0].values,
    "after_masked_constraint": masked_edit.iloc[0].values,
    "locked": [col in locked_features for col in FEATURE_COLS],
})
display(mask_table.round(3))

workflow_modes = pd.DataFrame([
    {"mode": "image-to-image", "classroom analog": "start from existing latent point", "primary risk": "unwanted drift from the source"},
    {"mode": "inpainting", "classroom analog": "lock selected attributes and regenerate others", "primary risk": "boundary or consistency mismatch"},
    {"mode": "outpainting", "classroom analog": "extend the candidate to a new format or channel", "primary risk": "extension does not match the original asset"},
    {"mode": "variation generation", "classroom analog": "sample many candidates under one condition", "primary risk": "diversity without brand discipline"},
])
display(workflow_modes)

## 10. Foundation-model ecosystem: retrieval, conditioning, generation, evaluation, and provenance

A diffusion model is rarely deployed alone. A language or planning component can translate a brief into structured conditions. Retrieval can fetch approved examples and policy context. The generator creates candidates. Evaluation gates and human review decide what moves forward. The next cells simulate this ecosystem without calling an external model.

In [ ]:
# ============================================================
# 10.1 Retrieve approved context for a brief
# ============================================================
brief = "minimal sustainable skincare retention calm product trust"
brief_vector = vectorizer.transform([brief])
distances, indices = retriever.kneighbors(brief_vector)
retrieved = concepts_df.iloc[indices[0]].copy()
retrieved["retrieval_distance"] = distances[0]
retrieved_approved = retrieved[retrieved["approved"]].copy()
if len(retrieved_approved) == 0:
    retrieved_approved = retrieved.copy()

retrieved_cols = [
    "concept_id", "product_category", "audience", "style", "objective",
    "brand_fit", "compliance_risk", "retrieval_distance", "concept_text"
]
display(retrieved_approved[retrieved_cols].round(3))

conditioning_package = {
    "brief": brief,
    "structured_request": REQUEST,
    "retrieved_concept_ids": retrieved_approved["concept_id"].head(5).tolist(),
    "brand_fit_threshold": 0.62,
    "compliance_risk_threshold": 0.46,
    "guidance_scale": 1.6,
    "num_denoising_steps": 30,
}
print(json.dumps(conditioning_package, indent=2))

In [ ]:
# ============================================================
# 10.2 Generate candidates and apply quality gates
# ============================================================
z_candidates = sample_ddim(
    request_cond,
    n_samples=80,
    guidance_scale=conditioning_package["guidance_scale"],
    num_steps=conditioning_package["num_denoising_steps"],
    seed=SEED + 777,
)
features_candidates = decode_latents(z_candidates)
brand_fit, risk = business_scores_from_features(features_candidates, **REQUEST)
prod_pred = product_clf.predict(z_candidates)
style_pred = style_clf.predict(z_candidates)

candidate_table = features_candidates.copy()
candidate_table.insert(0, "candidate_id", [f"M19_{i:03d}" for i in range(len(candidate_table))])
candidate_table["predicted_product"] = prod_pred
candidate_table["predicted_style"] = style_pred
candidate_table["brand_fit"] = brand_fit
candidate_table["compliance_risk"] = risk
candidate_table["gate_product"] = candidate_table["predicted_product"].eq(REQUEST["product"])
candidate_table["gate_style"] = candidate_table["predicted_style"].eq(REQUEST["style"])
candidate_table["gate_brand_fit"] = candidate_table["brand_fit"] >= conditioning_package["brand_fit_threshold"]
candidate_table["gate_compliance"] = candidate_table["compliance_risk"] <= conditioning_package["compliance_risk_threshold"]
gate_cols = ["gate_product", "gate_style", "gate_brand_fit", "gate_compliance"]
candidate_table["approved_by_gates"] = candidate_table[gate_cols].all(axis=1)

review_queue = candidate_table.sort_values(["approved_by_gates", "brand_fit", "compliance_risk"], ascending=[False, False, True]).head(12)
review_cols = [
    "candidate_id", "predicted_product", "predicted_style", "brand_fit", "compliance_risk",
    "gate_product", "gate_style", "gate_brand_fit", "gate_compliance", "approved_by_gates",
    "brightness", "whitespace", "product_focus", "sustainability_cue", "claim_strength"
]
display(review_queue[review_cols].round(3))

quality_gate_summary = candidate_table[gate_cols + ["approved_by_gates"]].mean().reset_index()
quality_gate_summary.columns = ["gate", "pass_rate"]
display(quality_gate_summary.round(3))

## 11. Business readiness, provenance, and saved artifacts

Generative systems should leave evidence behind. At minimum, a team should record what model family was used, which conditioning package was applied, which retrieved assets informed the request, what sampling settings were used, what gates were applied, and which candidates were approved for review. This is especially important when outputs may become customer-facing assets.

In [ ]:
# ============================================================
# 11.1 Scorecard, provenance record, and governance checklist
# ============================================================
scorecard = pd.DataFrame([
    {"metric_family": "fidelity", "metric": "mean_nn_distance_to_reference", "value": float(frontier_df.loc[frontier_df["num_denoising_steps"] == 30, "mean_nn_distance_to_reference"].iloc[0]), "desired_direction": "lower"},
    {"metric_family": "coverage", "metric": "reference_coverage_rate", "value": float(frontier_df.loc[frontier_df["num_denoising_steps"] == 30, "reference_coverage_rate"].iloc[0]), "desired_direction": "higher"},
    {"metric_family": "diversity", "metric": "generated_diversity", "value": float(frontier_df.loc[frontier_df["num_denoising_steps"] == 30, "generated_diversity"].iloc[0]), "desired_direction": "context dependent"},
    {"metric_family": "adherence", "metric": "product_adherence_rate", "value": float(guidance_df.loc[guidance_df["guidance_scale"] == 1.6, "product_adherence_rate"].iloc[0]), "desired_direction": "higher"},
    {"metric_family": "adherence", "metric": "style_adherence_rate", "value": float(guidance_df.loc[guidance_df["guidance_scale"] == 1.6, "style_adherence_rate"].iloc[0]), "desired_direction": "higher"},
    {"metric_family": "governance", "metric": "candidate_gate_approval_rate", "value": float(candidate_table["approved_by_gates"].mean()), "desired_direction": "context dependent"},
])
display(scorecard.round(4))

governance_checklist = pd.DataFrame([
    {"checkpoint": "Input controls", "question": "Did the request use an approved conditioning package and retrieved context?", "status": "demo complete"},
    {"checkpoint": "Sampling policy", "question": "Are step count, guidance strength, and seed recorded?", "status": "demo complete"},
    {"checkpoint": "Technical gate", "question": "Are generated feature values inside valid ranges?", "status": "demo complete"},
    {"checkpoint": "Adherence gate", "question": "Do candidates match requested product and style conditions?", "status": "demo complete"},
    {"checkpoint": "Brand and compliance gate", "question": "Do candidates pass brand-fit and compliance-risk thresholds?", "status": "demo complete"},
    {"checkpoint": "Human review", "question": "Has a responsible person approved customer-facing use?", "status": "not automated in notebook"},
])
display(governance_checklist)

provenance_record = {
    "chapter": "Chapter 19 Modern Generative Models",
    "model_family": "toy conditional diffusion in PCA latent space",
    "training_data": {
        "source": "synthetic creative concept library",
        "train_months": "1 to 18",
        "test_months": "19 to 24",
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
    },
    "latent_representation": {
        "method": "PCA proxy for latent diffusion",
        "latent_dims": LATENT_DIMS,
        "explained_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
        "reconstruction_rmse": pca_rmse,
    },
    "diffusion_policy": {
        "timesteps": T_STEPS,
        "training_epochs": EPOCHS_DIFFUSION,
        "condition_dropout": COND_DROPOUT,
        "guidance_scale": conditioning_package["guidance_scale"],
        "num_denoising_steps": conditioning_package["num_denoising_steps"],
        "seed": SEED,
    },
    "conditioning_package": conditioning_package,
    "quality_gates": {
        "brand_fit_threshold": conditioning_package["brand_fit_threshold"],
        "compliance_risk_threshold": conditioning_package["compliance_risk_threshold"],
        "required_product": REQUEST["product"],
        "required_style": REQUEST["style"],
    },
    "intended_use": "classroom demonstration of diffusion workflow controls, not production creative generation",
    "known_limits": [
        "The data are synthetic and low-dimensional.",
        "PCA is only a classroom proxy for a learned latent encoder and decoder.",
        "The classifier-based adherence checks are diagnostics, not legal or brand approval.",
        "Human review is required before customer-facing use in a real workflow.",
    ],
}

print(json.dumps(provenance_record, indent=2))

In [ ]:
# ============================================================
# 11.2 Save notebook artifacts
# ============================================================
concepts_df.to_csv(OUTPUT_DIR / "ch19_synthetic_creative_library.csv", index=False)
frontier_df.to_csv(OUTPUT_DIR / "ch19_speed_quality_frontier.csv", index=False)
guidance_df.to_csv(OUTPUT_DIR / "ch19_guidance_experiment.csv", index=False)
candidate_table.to_csv(OUTPUT_DIR / "ch19_generated_candidate_table.csv", index=False)
scorecard.to_csv(OUTPUT_DIR / "ch19_generation_scorecard.csv", index=False)
write_json(OUTPUT_DIR / "ch19_provenance_record.json", provenance_record)

saved_files = sorted([p.name for p in OUTPUT_DIR.iterdir()])
print("Saved artifacts:")
for name in saved_files:
    print(f"- {name}")

## Decision guide

Use a diffusion-style workflow when iterative refinement, controllability, and editing are central to the business problem. Use faster sampling settings for early ideation, and reserve stricter settings for review-ready candidates. Treat guidance strength as a governance parameter because it changes the adherence-diversity balance. Treat latent diffusion as a computational design pattern: generate in a compressed space, then decode, evaluate, and review. For customer-facing workflows, the generator should be surrounded by retrieval, quality gates, provenance logging, and human approval.

## Exercises

1. Change `guidance_scale` in Section 10 from `1.6` to `0.8` or `3.0`, rerun the generation and gate cells, and compare adherence, diversity, and approval rates.

2. Change `num_denoising_steps` in the conditioning package from `30` to `8` or `40`. Does a slower setting always improve the scorecard in this toy system?

3. Modify the `REQUEST` dictionary to generate a banking concept with an instructional style. Which quality gates become stricter, and why might compliance risk increase?

4. Add one more gate to `candidate_table`, such as `claim_strength <= 0.50` or `urgency_cue <= 0.60`. How does the review queue change?

5. Replace the TF-IDF retrieval query with your own creative brief. Inspect the retrieved concepts and explain how retrieval changes the conditioning package.

6. Increase `LATENT_DIMS` from `2` to `3`. Which sections must be updated for visualization, and how does reconstruction RMSE change?